In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample.to_csv('submission.csv', index=False)

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [4]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
!pip install -q chonkie sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.9/230.9 kB 5.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.2/387.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 58.5 MB/s eta 0:00:00:00:01


In [6]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def solve_mcq_tfidf(prompt, options):
    documents = [prompt] + options
  
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(documents)
    
    prompt_vector = tfidf_matrix[0:1]
    option_vectors = tfidf_matrix[1:]
    
    
    similarities = cosine_similarity(prompt_vector, option_vectors).flatten()
    
    
    labels = ['A', 'B', 'C', 'D', 'E']
    ranked_indices = np.argsort(similarities)[::-1]
    

    top_3 = [labels[i] for i in ranked_indices[:3]]
    return " ".join(top_3)


test_prompt = "What process do plants use to convert sunlight into food?"
test_options = [
    "Cellular Respiration", # A
    "Photosynthesis",       # B
    "Osmosis",              # C
    "Transpiration",        # D
    "Fermentation"          # E
]

prediction = solve_mcq_tfidf(test_prompt, test_options)
print(f"Top 3 Predictions: {prediction}") 

Top 3 Predictions: E D C


In [7]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2') 

def solve_mcq_transformer(row):
    prompt = str(row['prompt'])
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    prompt_embedding = model.encode([prompt])
    option_embeddings = model.encode(options)
    
    similarities = cosine_similarity(prompt_embedding, option_embeddings).flatten()
    
    ranked_indices = np.argsort(similarities)[::-1]
    
    top_3 = [labels[i] for i in ranked_indices[:3]]
    return " ".join(top_3)

print("Generating predictions...")
test_df['prediction'] = test_df.apply(solve_mcq_transformer, axis=1)

submission = test_df[['id', 'prediction']]
submission.to_csv('submission.csv', index=False)
print("Submission saved!")

Loading model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating predictions...
Submission saved!
